# Space Fault Recovery — GRPO Training Notebook

Trains `Qwen/Qwen2.5-1.5B-Instruct` with **HF TRL GRPO** on the Space Fault Recovery OpenEnv environment.

The agent is an LLM that reads spacecraft telemetry and must choose repair actions to recover a satellite from 1–3 injected faults within 50 steps.

**Recommended runtime:** A10G (24 GB VRAM) via HF Jobs or Colab Pro+.  
**Expected wall time:** ~35–45 min for 500 gradient steps.  
**Estimated HF credit cost:** ~\$1.50 on A10G.

---
**What this notebook does:**
1. Installs dependencies
2. Clones the repo and sets up sys.path
3. Runs a quick environment smoke test
4. Builds a mixed step-0 + mid-episode prompt dataset
5. Evaluates the base model (pre-training baseline)
6. Runs GRPO training with LoRA + WandB
7. Evaluates the trained model and compares to baseline
8. Saves reward curves as PNG plots

In [ ]:
# ── Step 1: Install dependencies ────────────────────────────────────────────
# Skip torch install if already present (Colab/HF Jobs has it pre-installed)
import subprocess, sys

packages = [
    "transformers>=4.40.0",
    "trl>=0.9.0",
    "peft>=0.10.0",
    "bitsandbytes>=0.43.0",
    "accelerate>=0.28.0",
    "datasets>=2.18.0",
    "wandb>=0.16.0",
    "matplotlib>=3.8.0",
    "openenv-core[core]>=0.2.2",
]

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + packages)
print("Dependencies installed.")

In [ ]:
# ── Step 2: WandB login ─────────────────────────────────────────────────────
# Paste your WandB API key when prompted, or set WANDB_API_KEY env variable.
import wandb
wandb.login()

In [ ]:
# ── Step 3: Clone repo and set up sys.path ───────────────────────────────────
import os, sys

REPO_URL = "https://huggingface.co/spaces/DivitTas/Space-Fault-Recovery"
REPO_DIR = "/content/space_fault_recovery"

if not os.path.exists(REPO_DIR):
    subprocess.check_call(["git", "clone", REPO_URL, REPO_DIR])

# Add repo root to path so trl_train.py can import models, server/, training/
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

In [ ]:
# ── Step 4: Environment smoke test ──────────────────────────────────────────
from training.openenv_compat import ensure_training_runtime
ensure_training_runtime()

from server.space_fault_recovery_environment import SpaceFaultRecoveryEnvironment
from models import SpaceFaultAction

env = SpaceFaultRecoveryEnvironment()
obs = env.reset(seed=42)
print(f"Reset OK — mission: {obs.mission_status}, battery: {obs.battery_pct:.1f}%")
print(f"Active faults: {env._sc.active_faults}")

obs2 = env.step(SpaceFaultAction(command="diagnostic_scan", target="power"))
print(f"Diagnostic result: {obs2.last_action_result}")
print(f"Reward: {obs2.reward:.3f}")
print("\nSmoke test passed.")

In [ ]:
# ── Step 5: Dataset preview ──────────────────────────────────────────────────
# Build a tiny dataset to verify the mid-episode prompt pipeline
import sys
sys.path.insert(0, REPO_DIR)

# Re-import after path setup
import importlib
import trl_train
importlib.reload(trl_train)

mini_dataset = trl_train.build_prompt_dataset(num_prompts=12, seed_base=1_000)
print(f"Columns: {mini_dataset.column_names}")
print(f"\nSample step-0 prompt (first 500 chars):\n{mini_dataset[0]['prompt'][:500]}")
print(f"\nSample mid-episode prompt (row 5, first 500 chars):\n{mini_dataset[5]['prompt'][:500]}")
import json
print(f"\nRow 5 prefix actions: {json.loads(mini_dataset[5]['prefix_actions_json'])}")

In [ ]:
# ── Step 6: Pre-training baseline ────────────────────────────────────────────
# The training script (Step 7) will run evaluate_policy() before training
# and save results to logs/grpo/<run-name>/eval_pretrain.json automatically.
# We load those results here after training completes for the comparison table.
# (No model load needed in this cell — avoids double-loading and OOM.)
print("Pre-training evaluation will run automatically inside trl_train.main().")
print("Results will be saved to logs/grpo/<run-name>/eval_pretrain.json")

In [ ]:
# ── Step 7: GRPO Training ───────────────────────────────────────────────────
# Recommended: A10G (24 GB) with LoRA, fp16, 500 steps.
# Adjust MAX_STEPS for a smoke run (e.g. 10) or a full run (500).

RUN_NAME = "grpo-space-fault-lora"   # change for each new run
MAX_STEPS = 500                       # set to 10 for a quick smoke run
USE_LORA = True                       # set False only if you have an A100
LOAD_IN_4BIT = False                  # set True for T4 (16 GB) runs

# Build sys.argv to pass args to trl_train.main()
import sys
sys.argv = [
    "trl_train.py",
    "--run-name", RUN_NAME,
    "--max-steps", str(MAX_STEPS),
    "--report-to", "wandb",
    "--wandb-project", "space-fault-trl",
    "--num-prompts", "256",
    "--eval-episodes", "20",
    "--log-dir", "logs/grpo",
    "--per-device-train-batch-size", "4",
    "--max-completion-length", "128",
]

if USE_LORA:
    sys.argv += ["--use-lora", "--lora-r", "16", "--lora-alpha", "32"]

if LOAD_IN_4BIT:
    sys.argv.append("--load-in-4bit")

trl_train.main()

In [ ]:
# ── Step 8: Before / after comparison ───────────────────────────────────────
import json
from pathlib import Path

log_dir = Path("logs/grpo") / RUN_NAME

pretrain_metrics = json.loads((log_dir / "eval_pretrain.json").read_text())
posttrain_metrics = json.loads((log_dir / "eval_posttrain.json").read_text())

print("=" * 50)
print(f"{'Metric':<25} {'Pre-train':>12} {'Post-train':>12}")
print("-" * 50)
for k in ["mean_reward", "success_rate", "min_reward", "max_reward"]:
    pre = pretrain_metrics.get(k, float("nan"))
    post = posttrain_metrics.get(k, float("nan"))
    delta = post - pre
    print(f"{k:<25} {pre:>12.3f} {post:>12.3f}  ({'+' if delta >= 0 else ''}{delta:.3f})")
print("=" * 50)

In [ ]:
# ── Step 9: Training curve plots ────────────────────────────────────────────
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

log_dir = Path("logs/grpo") / RUN_NAME
plots_dir = log_dir / "plots"
plots_dir.mkdir(parents=True, exist_ok=True)

csv_path = log_dir / "train_log.csv"
df = pd.read_csv(csv_path)
print(f"Training log: {len(df)} rows, columns: {df.columns.tolist()}")

# Reward curve (use the first reward-related column found)
reward_col = next(
    (c for c in df.columns if "reward" in c.lower() and "step" not in c.lower()),
    None,
)

if reward_col:
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(df["step"], df[reward_col], alpha=0.4, color="steelblue", label="per-step")
    rolling = df[reward_col].rolling(20, min_periods=1).mean()
    ax.plot(df["step"], rolling, color="steelblue", linewidth=2, label="20-step avg")
    ax.axhline(pretrain_metrics["mean_reward"], color="crimson", linestyle="--",
               linewidth=1.5, label=f"base model mean ({pretrain_metrics['mean_reward']:.2f})")
    ax.set_xlabel("Training step")
    ax.set_ylabel("Rollout reward")
    ax.set_title("GRPO Training Reward Curve — Space Fault Recovery")
    ax.legend()
    fig.tight_layout()
    path = plots_dir / "grpo_reward_curve.png"
    fig.savefig(path, dpi=150)
    print(f"Saved: {path}")
    plt.show()
else:
    print(f"No reward column found in: {df.columns.tolist()}")
    print("Check WandB for reward traces instead.")

# Loss curve
loss_col = next(
    (c for c in df.columns if "loss" in c.lower() and "step" not in c.lower()),
    None,
)
if loss_col:
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(df["step"], df[loss_col], alpha=0.4, color="darkorange", label="per-step")
    rolling_loss = df[loss_col].rolling(20, min_periods=1).mean()
    ax.plot(df["step"], rolling_loss, color="darkorange", linewidth=2, label="20-step avg")
    ax.set_xlabel("Training step")
    ax.set_ylabel("Loss")
    ax.set_title("GRPO Training Loss — Space Fault Recovery")
    ax.legend()
    fig.tight_layout()
    path = plots_dir / "grpo_loss_curve.png"
    fig.savefig(path, dpi=150)
    print(f"Saved: {path}")
    plt.show()

## Next Steps

- **Longer runs**: increase `--max-steps` to 1000–2000 for more pronounced improvement curves.
- **Upload trained adapter**: `model.push_to_hub("your-hf-username/space-fault-recovery-grpo")`
- **Link your WandB run** in the README so judges can view live metrics.
- **Blog post**: include the reward curve PNG, the before/after table, and a 2-minute explanation of what the agent learned.

The `logs/grpo/<run-name>/` directory contains:
- `config.json` — run hyperparameters
- `train_log.csv` — per-step reward/loss
- `eval_pretrain.json` — base model metrics
- `eval_posttrain.json` — trained model metrics
- `plots/grpo_reward_curve.png` — reward curve for README